Kvanttiteleportaatio

Olkoon Alice ja Bob, näillä on yksi yhteinen "bell-tila" S=1/sqrt(2)*(|0>_A x |0>_B + |1>_A x |1>_B). Olkoon Alicella jokin tuiki tuntematon tila a*|0>+b|1>, jonka hän haluaisi lähettää Bobille. Teleportaatio-osio toimii seuraavanlaisesti: Ensin CNOT tuntemattomasta tilasta S:ään (ts. S osaan asetetaan CNOT). Tämän jälkeen hadamard tuntemattomaan tilaan. --> mitataan, ja mittausten pohjalta määritellään mitä Bobin tulee tehdä rekonstruoidakseen alkuperäisen tilan, Alicen mittausten pohjalta (eli ts. mitä pitää tehdä lopussa).

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit import transpile, QuantumCircuit, ClassicalRegister
#from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
import random


def simulate_measurements(qc_samples, shots):
    backend = AerSimulator(seed_simulator = random.randint(0,10000000))
    qc_t = transpile(qc_samples, backend)
    job = backend.run(qc_t, shots=shots)
    return job.result()
# Applyataan suoraan olemassa olevaan circuittiin
def gen_bellstate(qc: QuantumCircuit, q1: int, q2: int, label: str):
    match label:
        case "00":
            pass
        case "01":
            qc.x(q2)
            qc.h(q1)
        case "10":
            qc.x(q1)
        case "11":
            qc.x(q1)
            qc.x(q2)
        case _:
            raise ValueError(f"Bell labels are: 00, 01, 10, 11, not {label}")
    qc.h(q1)
    qc.cx(q1, q2)

In [ ]:
qc = QuantumCircuit(2)
# a,b määritellään numeerisesti. Käytetään randomia:
a = random.uniform(0,1)
b = 1-a
creg_a = ClassicalRegister(2, "alice")
creg_b = ClassicalRegister(1, "bob")
qc = QuantumCircuit(3) # Tuntematon, bellstate_00
qc.add_register(creg_a)
qc.add_register(creg_b)
#qc.ry(theta,0)
#qc.rz(phi,0)
qc.initialize([a**0.5,b**0.5],0)
# Luodaan toisesta qubitista bellstate_00:
#qc.h(1)
#qc.cx(1, 2)
gen_bellstate(qc, 1, 2, "00")

init_state = Statevector(qc)

# Nyt itse teleportaatio:
qc.cx(0, 1)
qc.h(0)
# Alice mittaa
mid_state = Statevector(qc)
qc_samples = qc.copy()
qc_samples.measure([0,1], [creg_a[0], creg_a[1]])
shots = 1
alice_result = simulate_measurements(qc_samples, shots)
qubit_bitstring = list(alice_result.get_counts().keys())[0][2:]
# Loppukäsittely, olkoon mittaus muotoa nm, täten alimmaiseen qubittiin
# tehdään Z^n X^m operaatio:

match qubit_bitstring:
    case "00":
        pass
    case "01":
        qc.x(2)
    case "10":
        qc.z(2)
    case "11":
        qc.z(2)
        qc.x(2)
end_state = Statevector(qc)

print(f'''Counts (shots {shots}): {alice_result.get_counts()}
init_state: {init_state}
mid_state: {mid_state}
end_state: {end_state}''')
qc.draw("mpl")
